# CE49X – Real Estate Valuation Challenge
## Polynomial Regression + Random Forest + K-Fold CV + Hyperparameter Tuning

Bu notebook, **Taiwan Real Estate Valuation** veri seti ile farklı regresyon yaklaşımlarını kullanarak ev fiyatı tahmini yapmak için hazırlanmıştır.

Amaç: **Mümkün olduğunca yüksek ve güvenilir bir R² skoru** elde etmek ve yöntemleri birbirleriyle karşılaştırmak.

Modeller:
- PolynomialFeatures + LinearRegression (degree = 1–5)
- RandomForestRegressor (hyperparameter tuning ile)

Değerlendirme yöntemleri:
- Farklı **train–test split** oranları ile hold-out: test_size = 0.2, 0.25, 0.3, 0.6
- **5-fold cross-validation (K-Fold CV)**

En sonunda:
- En iyi polinomsal modeli (degree + yöntem) belirliyoruz,
- Hyperparameter-tuned Random Forest ile karşılaştırıyoruz,
- Hangisi daha iyi R² veriyorsa onu **nihai model** olarak seçiyoruz.


## 1. Kütüphanelerin Yüklenmesi

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

from sklearn.model_selection import (
    train_test_split,
    KFold,
    cross_val_score,
    GridSearchCV
)
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error
)

# Grafik ayarları
plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['axes.grid'] = True

RANDOM_STATE = 42  # Reprodüksiyon için sabit rastgelelik tohumu


## 2. Veri Setinin Yüklenmesi

Veri seti ödev açıklamasına göre `challenge/data/Real estate valuation data set.xlsx` yolunda bulunuyor. Önce bu yolu deniyoruz; eğer yoksa notebook ile aynı klasörde arıyoruz.


In [ ]:
# Veri dosya yolunu ayarla – gerekirse burayı kendi klasör yapına göre değiştir
data_path = Path('challenge/data/Real estate valuation data set.xlsx')

if not data_path.exists():
    # Alternatif: notebook ile aynı klasördeyse
    alt_path = Path('Real estate valuation data set.xlsx')
    if alt_path.exists():
        data_path = alt_path

print(f'Kullanılan veri yolu: {data_path}')

# Excel dosyasını yükle
df = pd.read_excel(data_path)

# Kolon isimlerindeki gereksiz boşlukları temizleyelim
df.columns = df.columns.str.strip()

df.head()

## 3. Keşifsel Veri Analizi (EDA)

Veri setinin boyutu, veri tipleri, eksik değerler ve hedef değişkenin temel istatistiklerine bakıyoruz.


In [ ]:
print('Veri seti boyutu:', df.shape)
print('\nVeri tipleri:')
print(df.dtypes)

print('\nEksik değer sayıları:')
print(df.isna().sum())

target_col = 'Y house price of unit area'
print('\nHedef değişken temel istatistikleri:')
print(df[target_col].describe())

In [ ]:
# Hedef değişken dağılımı
plt.hist(df[target_col], bins=20)
plt.title('Y house price of unit area dağılımı')
plt.xlabel('Fiyat (10,000 NTD / Ping)')
plt.ylabel('Frekans')
plt.show()

# Sayısal özellikler için korelasyon matrisi
numeric_df = df.select_dtypes(include=[np.number])
corr = numeric_df.corr()

fig, ax = plt.subplots(figsize=(8, 6))
cax = ax.imshow(corr, cmap='coolwarm', vmin=-1, vmax=1)
ax.set_title('Sayısal Değişkenler Korelasyon Matrisi')
fig.colorbar(cax, ax=ax, fraction=0.046, pad=0.04)

ax.set_xticks(range(len(corr.columns)))
ax.set_yticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=45, ha='right')
ax.set_yticklabels(corr.columns)

plt.tight_layout()
plt.show()

## 4. Özellik (X) ve Hedef (y) Ayrımı

- `No` kolonu yalnızca satır index'i işlevi görüyor, modelde kullanılmıyor.
- Özellikler: `X1`–`X6` sütunları
- Hedef: `Y house price of unit area`


In [ ]:
# 'No' kolonunu düşür (varsa)
if 'No' in df.columns:
    df = df.drop(columns=['No'])

feature_cols = [
    'X1 transaction date',
    'X2 house age',
    'X3 distance to the nearest MRT station',
    'X4 number of convenience stores',
    'X5 latitude',
    'X6 longitude'
]

X = df[feature_cols].copy()
y = df[target_col].copy()

print('Özellik matrisi boyutu:', X.shape)
print('Hedef vektör boyutu:', y.shape)
X.head()

## 5. Polinomsal Regresyon Modeli

Her derece (degree = 1–5) için aşağıdaki pipeline'ı kullanıyoruz:

1. `PolynomialFeatures(degree=d, include_bias=False)` – Polinom ve etkileşim terimleri üretir.
2. `LinearRegression()` – Bu yeni özellikler üzerinde doğrusal regresyon uygular.


In [ ]:
def build_polynomial_model(degree: int) -> Pipeline:
    """Verilen degree için PolynomialFeatures + LinearRegression pipeline'ı döndürür."""
    model = Pipeline([
        ('poly', PolynomialFeatures(degree=degree, include_bias=False)),
        ('linreg', LinearRegression())
    ])
    return model


## 6. Polinomsal Regresyon – Farklı Train–Test Oranları ile Hold-Out Değerlendirme

Kombinasyonlar:
- Degree: 1, 2, 3, 4, 5
- Test oranı: 0.2, 0.25, 0.3, 0.6

Her kombinasyon için hesaplanan metrikler:
- Train R²
- Test R²
- Test MAE
- Test RMSE


In [ ]:
degrees = [1, 2, 3, 4, 5]
test_sizes = [0.2, 0.25, 0.3, 0.6]

holdout_results = []

for degree in degrees:
    for test_size in test_sizes:
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=test_size, random_state=RANDOM_STATE
        )

        model = build_polynomial_model(degree)
        model.fit(X_train, y_train)

        y_train_pred = model.predict(X_train)
        y_test_pred = model.predict(X_test)

        train_r2 = r2_score(y_train, y_train_pred)
        test_r2 = r2_score(y_test, y_test_pred)
        test_mae = mean_absolute_error(y_test, y_test_pred)
        test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))

        holdout_results.append({
            'degree': degree,
            'test_size': test_size,
            'train_r2': train_r2,
            'test_r2': test_r2,
            'test_mae': test_mae,
            'test_rmse': test_rmse
        })

holdout_df = pd.DataFrame(holdout_results)

print('Hold-out sonuçları (degree & test_size kombinasyonları):')
holdout_df

In [ ]:
# Test R²'e göre en iyi hold-out sonucu
best_holdout = holdout_df.sort_values('test_r2', ascending=False).iloc[0]
print('En iyi HOLD-OUT sonucu:')
print(best_holdout)

# Görsel olarak da test R² karşılaştıralım
fig, ax = plt.subplots()
for degree in degrees:
    subset = holdout_df[holdout_df['degree'] == degree]
    ax.plot(subset['test_size'], subset['test_r2'], marker='o', label=f'Degree {degree}')

ax.set_title('Polynomial Regression – Farklı Test Oranları için Test R²')
ax.set_xlabel('Test oranı')
ax.set_ylabel('Test R²')
ax.legend(title='Degree')
plt.show()

## 7. Polinomsal Regresyon – 5-Fold Cross-Validation

Bu bölümde degree = 1–5 için 5-fold cross-validation uyguluyoruz ve her degree için:
- Ortalama CV R²
- R² standart sapması
- Ortalama CV RMSE
- RMSE standart sapması
hesaplıyoruz.


In [ ]:
k = 5
kf = KFold(n_splits=k, shuffle=True, random_state=RANDOM_STATE)

cv_results = []

for degree in degrees:
    model = build_polynomial_model(degree)

    # R² skorları
    r2_scores = cross_val_score(
        model, X, y,
        cv=kf,
        scoring='r2'
    )

    # Negatif MSE skorları (RMSE için)
    neg_mse_scores = cross_val_score(
        model, X, y,
        cv=kf,
        scoring='neg_mean_squared_error'
    )

    mse_scores = -neg_mse_scores
    rmse_scores = np.sqrt(mse_scores)

    cv_results.append({
        'degree': degree,
        'cv_mean_r2': r2_scores.mean(),
        'cv_std_r2': r2_scores.std(),
        'cv_mean_rmse': rmse_scores.mean(),
        'cv_std_rmse': rmse_scores.std()
    })

cv_df = pd.DataFrame(cv_results)

print('5-Fold Cross-Validation sonuçları (Polynomial Regression):')
cv_df

In [ ]:
# CV R²'e göre en iyi degree
best_poly_cv = cv_df.sort_values('cv_mean_r2', ascending=False).iloc[0]
print('En iyi POLINOMSAL model (5-fold CV):')
print(best_poly_cv)

# Degree'lere göre ortalama CV R² grafiği
fig, ax = plt.subplots()
ax.plot(cv_df['degree'], cv_df['cv_mean_r2'], marker='o')
ax.set_title('Polynomial Regression – Degree vs Ortalama CV R² (5-fold)')
ax.set_xlabel('Degree')
ax.set_ylabel('Ortalama CV R²')
plt.show()

# Degree'lere göre ortalama CV RMSE grafiği
fig, ax = plt.subplots()
ax.plot(cv_df['degree'], cv_df['cv_mean_rmse'], marker='o')
ax.set_title('Polynomial Regression – Degree vs Ortalama CV RMSE (5-fold)')
ax.set_xlabel('Degree')
ax.set_ylabel('Ortalama CV RMSE')
plt.show()

## 8. Random Forest Regresyonu – Hyperparameter Tuning ve Değerlendirme

Şimdi, özellikleri olduğu gibi kullanarak (polinomsuz) bir **RandomForestRegressor** modeli kuracağız.

Adımlar:
1. Basit bir Random Forest ile 5-fold CV sonucu (baseline)
2. Hyperparameter grid tanımlayarak **GridSearchCV** ile en iyi hiperparametreleri bulma
3. En iyi Random Forest modeli için 5-fold CV R² ve RMSE değerlerini raporlama
4. Seçili bir hold-out split (test_size=0.2) üzerinde performansını ölçme


In [ ]:
# 8.1 Baseline Random Forest (default hyperparameters)
rf_base = RandomForestRegressor(random_state=RANDOM_STATE)

rf_base_r2_scores = cross_val_score(
    rf_base, X, y,
    cv=kf,
    scoring='r2'
)

rf_base_neg_mse_scores = cross_val_score(
    rf_base, X, y,
    cv=kf,
    scoring='neg_mean_squared_error'
)
rf_base_mse_scores = -rf_base_neg_mse_scores
rf_base_rmse_scores = np.sqrt(rf_base_mse_scores)

print('Baseline Random Forest – 5-fold CV:')
print(f'Ortalama R²:  {rf_base_r2_scores.mean():.4f} ± {rf_base_r2_scores.std():.4f}')
print(f'Ortalama RMSE: {rf_base_rmse_scores.mean():.4f} ± {rf_base_rmse_scores.std():.4f}')

In [ ]:
# 8.2 Hyperparameter Grid ve GridSearchCV
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 5, 10, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['auto', 'sqrt', 'log2']
}

rf = RandomForestRegressor(random_state=RANDOM_STATE)

grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=kf,
    scoring='r2',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X, y)

best_rf = grid_search.best_estimator_
best_rf_cv_r2 = grid_search.best_score_

print('\nEn iyi Random Forest hiperparametreleri:')
print(grid_search.best_params_)
print(f'En iyi CV R² (Random Forest): {best_rf_cv_r2:.4f}')

In [ ]:
# 8.3 En iyi RF modeli için RMSE'yi de CV ile hesaplayalım
best_rf_neg_mse_scores = cross_val_score(
    best_rf, X, y,
    cv=kf,
    scoring='neg_mean_squared_error'
)
best_rf_mse_scores = -best_rf_neg_mse_scores
best_rf_rmse_scores = np.sqrt(best_rf_mse_scores)

print('En iyi Random Forest – 5-fold CV (tekrar hesaplanan):')
print(f'Ortalama R²:  {best_rf_cv_r2:.4f}')
print(f'Ortalama RMSE: {best_rf_rmse_scores.mean():.4f} ± {best_rf_rmse_scores.std():.4f}')

In [ ]:
# 8.4 En iyi RF modelini sabit bir hold-out split (test_size=0.2) üzerinde değerlendirelim
X_train_rf, X_test_rf, y_train_rf, y_test_rf = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

best_rf.fit(X_train_rf, y_train_rf)
y_train_rf_pred = best_rf.predict(X_train_rf)
y_test_rf_pred = best_rf.predict(X_test_rf)

rf_train_r2 = r2_score(y_train_rf, y_train_rf_pred)
rf_test_r2 = r2_score(y_test_rf, y_test_rf_pred)
rf_test_mae = mean_absolute_error(y_test_rf, y_test_rf_pred)
rf_test_rmse = np.sqrt(mean_squared_error(y_test_rf, y_test_rf_pred))

print('Hyperparametre ayarlı Random Forest – Hold-Out (test_size=0.2):')
print(f'Train R²: {rf_train_r2:.4f}')
print(f'Test  R²: {rf_test_r2:.4f}')
print(f'Test  MAE: {rf_test_mae:.4f}')
print(f'Test  RMSE: {rf_test_rmse:.4f}')

## 9. Polynomial vs Random Forest – En İyi Modelin Seçilmesi

Artık elimizde:
- Polinomsal regresyon için en iyi 5-fold CV sonucu (`best_poly_cv`)
- Hyperparameter-tuned Random Forest için en iyi 5-fold CV sonucu (`best_rf_cv_r2`)

Bu bölümde bu iki yöntemi **CV R²** değerleri üzerinden karşılaştırıyoruz ve hangisi yüksekse onu **nihai model** olarak seçiyoruz.


In [ ]:
best_poly_cv_r2 = best_poly_cv['cv_mean_r2']

print('Polinomsal regresyon – en iyi degree ve CV R²:')
print(best_poly_cv)
print(f"\nPolynomial BEST CV R²: {best_poly_cv_r2:.4f}")
print(f"Random Forest BEST CV R²: {best_rf_cv_r2:.4f}")

if best_rf_cv_r2 >= best_poly_cv_r2:
    final_model_type = 'random_forest'
    final_model = best_rf
    print('\nGenel olarak EN İYİ model: Random Forest (hyperparameter tuned)')
else:
    final_model_type = 'polynomial'
    final_degree = int(best_poly_cv['degree'])
    final_model = build_polynomial_model(final_degree)
    final_model.fit(X, y)
    print('\nGenel olarak EN İYİ model: Polynomial Regression')
    print(f'Degree = {final_degree}, Ortalama CV R² = {best_poly_cv_r2:.4f}')

In [ ]:
# Eğer final model Random Forest ise, tam veri üzerinde tekrar eğitelim
if final_model_type == 'random_forest':
    final_model.fit(X, y)
    print('Nihai Random Forest modeli, tüm veri üzerinde eğitildi.')
else:
    print('Nihai Polynomial Regression modeli, önceki hücrede zaten tüm veri üzerinde eğitildi.')

## 10. Nihai Modelin Özellik Etkilerinin Analizi

Nihai modelin türüne göre farklı analiz yapıyoruz:
- **Polynomial Regression** ise: PolynomialFeatures ile üretilen tüm terimler için katsayı analizi.
- **Random Forest** ise: Özellik önemleri (`feature_importances_`).


In [ ]:
if final_model_type == 'polynomial':
    poly_step = final_model.named_steps['poly']
    linreg_step = final_model.named_steps['linreg']

    poly_feature_names = poly_step.get_feature_names_out(feature_cols)
    coefficients = linreg_step.coef_

    coef_df = pd.DataFrame({
        'feature': poly_feature_names,
        'coefficient': coefficients
    })
    coef_df['abs_coefficient'] = coef_df['coefficient'].abs()
    coef_df_sorted = coef_df.sort_values('abs_coefficient', ascending=False)

    print('Polinomsal model – en büyük katsayıya sahip ilk 10 terim:')
    display(coef_df_sorted.head(10))

    # MRT içeren terimler
    mrt_mask = coef_df['feature'].str.contains('X3 distance to the nearest MRT station')
    mrt_terms = coef_df[mrt_mask].sort_values('abs_coefficient', ascending=False)
    print("\nMRT mesafesiyle ilgili terimler:")
    display(mrt_terms.head(10))

else:
    # Random Forest feature importances
    importances = final_model.feature_importances_
    fi_df = pd.DataFrame({
        'feature': feature_cols,
        'importance': importances
    }).sort_values('importance', ascending=False)

    print('Random Forest – Özellik Önemleri:')
    display(fi_df)

    plt.barh(fi_df['feature'], fi_df['importance'])
    plt.gca().invert_yaxis()
    plt.title('Random Forest – Feature Importances')
    plt.xlabel('Önem')
    plt.ylabel('Özellik')
    plt.show()

### MRT Mesafesi Yorumu

- Eğer **Polynomial Regression** seçildiyse ve MRT mesafesi içeren terimlerin katsayıları genellikle negatif ise: MRT uzaklığı arttıkça fiyatın azaldığı sonucuna varabiliriz.
- Eğer **Random Forest** seçildiyse ve `X3 distance to the nearest MRT station` özelliğinin önem skoru yüksekse: model, fiyat tahmininde MRT mesafesini önemli bir belirleyici olarak kullanıyor demektir.


## 11. Örnek Bir Konut İçin Fiyat Tahmini (Nihai Model ile)

Özellikler:
- House age = 5 yıl
- Distance to MRT = 500 metre
- Number of convenience stores = 3
- Transaction date = 2013.5
- Latitude ve longitude = veri setindeki medyan değerler


In [ ]:
# Latitude ve longitude için medyan değerler
lat_median = X['X5 latitude'].median()
lon_median = X['X6 longitude'].median()

print('Medyan enlem (latitude):', lat_median)
print('Medyan boylam (longitude):', lon_median)

# Örnek konut
example_house = pd.DataFrame({
    'X1 transaction date': [2013.5],
    'X2 house age': [5.0],
    'X3 distance to the nearest MRT station': [500.0],
    'X4 number of convenience stores': [3],
    'X5 latitude': [lat_median],
    'X6 longitude': [lon_median]
})

predicted_price = final_model.predict(example_house)[0]
print(f'Nihai model tahmini (Y house price of unit area): {predicted_price:.2f} (10,000 NTD / Ping)')

## 12. Kısa Özet

Bu notebookta:
1. Real Estate veri setini yükleyip temel EDA yaptık.
2. X1–X6 özelliklerini ve hedef değişkeni ayırdık.
3. Polynomial Regression için degree = 1–5 ve test_size = 0.2, 0.25, 0.3, 0.6 kombinasyonlarıyla hold-out sonuçlarını hesapladık.
4. Aynı polinomsal modeller için 5-fold CV ile ortalama R² ve RMSE değerlerini çıkardık.
5. RandomForestRegressor için önce baseline, sonra GridSearchCV ile hyperparameter tuning yaptık ve 5-fold CV sonuçlarını elde ettik.
6. En iyi Polynomial vs en iyi Random Forest modelini CV R² üzerinden karşılaştırıp genel olarak daha iyi olanı **nihai model** olarak seçtik.
7. Nihai modelin özellik etkilerini (katsayılar veya feature importance) analiz ettik.
8. Verilen örnek bir konut için fiy at tahmini yaptık.

Bu yapı sayesinde raporunda:
- Hem **model çeşitliliği** (linear/polynomial vs tree-based),
- Hem de **değerlendirme çeşitliliği** (hold-out vs K-Fold CV)
üzerinden oldukça güçlü bir analiz sunabilirsin.
